# 02_data_cleaning_and_resampling.ipynb

This notebook cleans the raw downloaded datasets and prepares them for feature engineering.

### Objectives:
1. Clean raw klines (duplicates, invalid values, outliers, timezone UTC conversion).
2. Perform data quality checks and generate reports under `reports/data_quality/`.
3. Align/merge futures metrics with kline datasets.
4. Save cleaned and processed dataframes in the `data/processed/` directory.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations

In [ ]:
from utils.config import load_global_config

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")
market_type = config.get("data", "market_type", "swap")

## 3. Clean OHLCV Data & Save Quality Reports

In [ ]:
from data.parquet_store import ParquetStore
from data.data_cleaning import clean_ohlcv, save_quality_report
from utils.time_utils import convert_to_utc

store = ParquetStore(PROJECT_ROOT)
timeframes = ["5m", "15m", "1h", "4h", "1d"]

for tf in timeframes:
    df_raw = store.load_raw_klines(symbol, tf)
    if df_raw is not None and not df_raw.empty:
        # Clean data
        df_clean, report = clean_ohlcv(df_raw)
        
        # Convert to UTC datetime index
        df_clean = convert_to_utc(df_clean, "timestamp")
        
        # Save cleaned data
        store.save_processed_klines(df_clean, symbol, tf)
        
        # Save quality report
        report_path = os.path.join(PROJECT_ROOT, "reports", "data_quality", f"{symbol}_{tf}_quality_report.json")
        save_quality_report(report, report_path)
        print(f"Finished cleaning {tf}. Final rows: {len(df_clean)}")
    else:
        print(f"Raw data not found for {tf}.")

## 4. Align Futures Metrics with the Primary 5m Timeframe

In [ ]:
from utils.io_utils import load_parquet, save_parquet

if market_type == "swap":
    # Load 5m clean OHLCV
    df_5m = store.load_processed_klines(symbol, "5m")
    
    if df_5m is not None:
        # Load and align funding rates
        funding_path = os.path.join(PROJECT_ROOT, "data", "raw", "futures_metrics", f"{symbol}_funding_rates.parquet")
        df_funding = load_parquet(funding_path)
        if df_funding is not None and not df_funding.empty:
            df_funding = convert_to_utc(df_funding, "timestamp")
            # Funding rates are set every 8h; forward fill them onto the 5m index
            df_5m = df_5m.join(df_funding, how="left")
            df_5m["funding_rate"] = df_5m["funding_rate"].ffill().fillna(0.0)
            
        # Load and align open interest
        oi_path = os.path.join(PROJECT_ROOT, "data", "raw", "futures_metrics", f"{symbol}_open_interest.parquet")
        df_oi = load_parquet(oi_path)
        if df_oi is not None and not df_oi.empty:
            df_oi = convert_to_utc(df_oi, "timestamp")
            df_5m = df_5m.join(df_oi, how="left")
            df_5m["open_interest"] = df_5m["open_interest"].ffill()
            df_5m["open_interest_value"] = df_5m["open_interest_value"].ffill()
            
        # Save aligned 5m data
        store.save_processed_klines(df_5m, symbol, "5m")
        print("Successfully aligned futures metrics with the 5m OHLCV dataset.")
else:
    print("Spot market selected; skipping futures metrics alignment.")

## Summary of Cleaned Artifacts

The following cleaned and aligned datasets have been saved:
1. `data/processed/BTCUSDT_{timeframe}_clean.parquet` (for 5m, 15m, 1h, 4h, 1d)
2. `reports/data_quality/BTCUSDT_{timeframe}_quality_report.json`

**Next Step**: Run [03_feature_engineering_market_structure.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/03_feature_engineering_market_structure.ipynb) to begin generating market structure features.